# Epsilon Fund - Walk-Forward Validation (Pairs Trading)
Uses `infrastructure/walk_forward/` to run rolling Optuna optimisation and evaluate OOS robustness for **spread-based pairs strategies**. Ideal to use after finding a pair relationship that seems to work in the backtesting framework, to ensure the logic generalises across time.

---
### Iteration Guidelines

**Overfitting the iteration process:** Each time you inspect OOS results and adjust parameters, you leak OOS information into your design decisions. Cap yourself at **3–4 iterations** — first run with everything free, second with obvious fixes from CV + plateau analysis, third to tighten remaining params. 

If the strategy still shows heavy overfitting signals after three passes, **the problem is the strategy architecture, not the parameters**. For pairs trading this usually means the cointegration relationship is non-stationary or the spread construction is wrong — no amount of z-score tuning will fix that.

**WFE:** Walk-forward efficiency — examine IS/OOS ratio (simplest). For mean-reversion pairs strategies, WFE > 0.5 is a reasonable bar; below 0.3 suggests the spread relationship doesn't persist OOS.

**Perturbation degradation:** Examine perturbation table to see if degradation reduces across runs.

| Signal | Meaning | Action |
|--------|---------|--------|
| IS Sharpe drops, OOS Sharpe holds or rises, WFE improves | Removing noise-fitting degrees of freedom | Continue iterating |
| Perturbation degradation shrinks across iterations | Parameters becoming more robust | Continue iterating |
| N/A plateau params decreasing across iterations | Strategy becoming more tolerant of parameter movement | Continue iterating |
| WFE improvement flattens (e.g. 0.55 → 0.65 → 0.67) | Diminishing returns — further fixes won't help much | Stop iterating |
| IS and OOS both decline but WFE rises (IS falls faster) | Constraining away real signal, not just noise | Stop iterating |
| OOS Sharpe keeps declining despite "better" param setup | Overfitting the iteration process itself | Stop — problem is strategy architecture, not parameters |
| WFE decreases after fixing a parameter | Locked in a param that was legitimately adapting across folds | Unfix that parameter and re-run |

**Pairs-specific red flags:**
| Signal | Meaning |
|--------|---------|
| Beta CV > 0.3 across folds | Hedge ratio unstable — relationship may not be cointegrated |
| OOS trades cluster in 1–2 folds, zero in others | Spread only mean-reverts in specific regimes |
| Lookback CV is high but z_lookback CV is low | Model fitting noise in beta estimation while z-score normalisation is stable — consider fixing lookback |

---


In [48]:
import sys, os
import pandas as pd
import numpy as np

ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))

from binance_client import get_binance_client, get_data
from wf_engine import walk_forward, plateau_analysis, plateau_summary, perturbation_test, cost_stress_test
from wf_visualizer import plot_walk_forward_results, plot_plateau_analysis

print("✓ All modules loaded")



✓ All modules loaded


---
## Data

**Pair construction** — fetch two assets independently and merge on their datetime index. The strategy trades the *spread* between Y (dependent) and X (independent), not either asset alone.

**Assets** — any Binance pair in `BASEQUOTE` format (e.g. `MATICUSDT`, `APTUSDT`, `ETHUSDT`).  
Verify availability at [binance.com/en/trade](https://www.binance.com/en/trade).

**Intervals** — `'1m'` `'5m'` `'15m'` `'1h'` `'4h'` `'1d'` `'1w'`

**Lookback** — days of history: must be >= `(burn_in + train_bars + test_bars) × n_folds`. For daily bars with 400 train / 85 test / 200 burn-in, ~2000+ days recommended.

**`merge_pair()`** — inner-joins two OHLCV frames on datetime, producing `Close_Y`, `Close_X` (and optionally `Volume_Y`, `Volume_X`). Rows where either asset has no bar are dropped, so the merged length may be shorter than either input.



In [49]:
# ── pair definition ───────────────────────────────────────────────────────────
SYMBOL_Y = 'UNIUSDT'   # dependent  (Y leg)
SYMBOL_X = 'SUSHIUSDT'     # independent (X leg)
INTERVAL = '1d'
LOOKBACK = 1500         # days of history

client = get_binance_client()

df_y = get_data(client, SYMBOL_Y, INTERVAL, LOOKBACK)
df_x = get_data(client, SYMBOL_X, INTERVAL, LOOKBACK)

print(f'{SYMBOL_Y}: {df_y.index[0].date()} → {df_y.index[-1].date()}  ({len(df_y)} bars)')
print(f'{SYMBOL_X}: {df_x.index[0].date()} → {df_x.index[-1].date()}  ({len(df_x)} bars)')

# ── merge into single frame ──────────────────────────────────────────────────
def merge_pair(df_y: pd.DataFrame, df_x: pd.DataFrame) -> pd.DataFrame:
    """Inner-join two OHLCV frames → Close_Y, Close_X aligned by datetime."""
    y = df_y[['Close']].rename(columns={'Close': 'Close_Y'})
    x = df_x[['Close']].rename(columns={'Close': 'Close_X'})
    merged = y.join(x, how='inner').dropna()
    return merged

df = merge_pair(df_y, df_x)
print(f'\nMerged pair: {df.index[0].date()} → {df.index[-1].date()}  ({len(df)} bars)')
df.tail(3)


UNIUSDT: 2022-03-07 → 2026-04-14  (1500 bars)
SUSHIUSDT: 2022-03-07 → 2026-04-14  (1500 bars)

Merged pair: 2022-03-07 → 2026-04-14  (1500 bars)


,Close_Y,Close_X
Time,,
2026-04-12,3.018,0.1889
2026-04-13,3.234,0.1989
2026-04-14,3.168,0.1966


---
## Parameter Configuration

Define which parameters to optimise and which to anchor — **recommended to do after strategy writeup and an initial free run**.

**Pairs-specific parameters typically include:**
| Parameter | Role | Typical Range |
|-----------|------|---------------|
| `lookback` | Rolling OLS window for beta/alpha estimation | 60–200 |
| `z_lookback` | Rolling window for z-score normalisation of the spread | 20–120 |
| `entry` | Z-score threshold to open a position | 1.0–3.0 |
| `exit` | Z-score threshold to close a position | 0.1–1.5 |
| `stop_z` | Hard z-score stop-loss | 3.0–6.0 |
| `max_holding` | Maximum bars to hold before forced exit | 5–40 |

`FIXED_PARAMS`: choose parameters with CV < 0.15 from prior stability run, cross-referencing with perturbation verdict table to reduce search space, improve OOS credibility.





In [50]:
# ── search ranges ─────────────────────────────────────────────────────────────
# Format: 'param_name': ('int' | 'float', lo, hi)
PARAM_DEFS = {
    'lookback':    ('int',   80,  160),    # rolling OLS window
    'z_lookback':  ('int',   20,  80),    # z-score normalisation window
    'entry':       ('float', 1, 3),    # |z| > entry → open
    'exit_z':      ('float', 0.1, 1.5),    # |z| < exit  → close
    'stop_z':      ('float', 3.5, 5.0),    # |z| > stop  → hard exit
    'max_holding': ('int',   5,   30),     # forced exit after N bars
}

# ── fixed params ──────────────────────────────────────────────────────────────
# Bypasses Optuna — populate after a stability run (CV < 0.15)
FIXED_PARAMS = {
    # 'lookback': 120,
    # 'stop_z':   4.0,
}



### *Guide to parameter anchoring*

|  | Robust Plateau | Fragile Plateau |
|--|----------------|-----------------|
| **Low CV** | Stable across folds and insensitive to exact value — keep free | Looks stable but is fitting the same noise patterns — fix at consensus |
| **High CV** | Parameter unimportant — fix at any reasonable value | Unstable across folds and sitting on a cliff — strong candidate to eliminate |

**Pairs-specific anchoring notes:**
- `lookback` (OLS window) often has high CV because the hedge ratio genuinely shifts over multi-year samples. If plateau is robust, let it stay free — the optimizer is adapting to structural shifts.
- `entry` / `exit_z` tend to be the most fragile — small changes in threshold can flip entire trade sequences. If CV is high *and* plateau is narrow, fix one of them.
- `stop_z` is usually insensitive (most trades exit before hitting it). If CV < 0.15, fix it early to free up search budget.

Copy-paste plateau analysis table above into fixed params section and decide manually on which to fix/keep free.


---
## Strategy

**CRUCIAL** — Strategy logic needs to work well in the backtesting notebook before running here; saves time not running walk-forward for a broken strategy.

**Pairs strategy pipeline:**
1. **Spread construction** — rolling OLS of `ln(Y)` on `ln(X)` → `beta`, `alpha`, then `spread = ln(Y) - alpha - beta * ln(X)`
2. **Z-score normalisation** — rolling mean/std of the spread over `z_lookback` → `z = (spread - mean) / std`
3. **Signal generation** — state machine: enter at `|z| > entry`, exit at `|z| < exit_z`, hard stop at `|z| > stop_z`, forced exit after `max_holding` bars
4. **Spread return** — `r_spread[t] = r_Y[t] - beta[t-1] * r_X[t]` (lagged beta to avoid lookahead)
5. **Synthetic close** — `Close[t] = Close[t-1] * (1 + position[t-1] * r_spread[t])` for PnL tracking

**Available columns in `df_slice`:** `Close_Y`, `Close_X` (from `merge_pair`)

**Required output:** a `position` column — `1` long spread · `0` flat · `-1` short spread  
**Optional:** `position_size` column (0–1) for fractional capital

> The engine applies a 1-bar execution lag automatically. Inside the strategy loop, use `prev` for signal decisions and `curr` for execution — no manual shifting needed.

**Lookahead bias checklist for pairs:**
- Beta used for signal at bar `t` must be computed from bars `≤ t-1`
- Z-score at bar `t` uses spread values through bar `t` (spread itself is contemporaneous, that's fine)
- Spread *return* at bar `t` uses `beta[t-1]`, not `beta[t]`



In [51]:
def my_strategy(df_slice: pd.DataFrame, params: dict):

    df = df_slice.copy()

    lookback    = int(params['lookback'])
    z_lookback  = int(params['z_lookback'])
    entry       = params['entry']
    exit_z      = params['exit_z']
    stop_z      = params['stop_z']
    max_holding = int(params['max_holding'])

    ln_y = np.log(df['Close_Y'])
    ln_x = np.log(df['Close_X'])

    beta = ln_y.rolling(lookback).cov(ln_x) / ln_x.rolling(lookback).var()
    alpha = ln_y.rolling(lookback).mean() - beta * ln_x.rolling(lookback).mean()

    df['beta'] = beta
    df['alpha'] = alpha

    df['spread'] = ln_y - (alpha + beta * ln_x)
    df['spread_mean'] = df['spread'].rolling(z_lookback).mean()
    df['spread_std'] = df['spread'].rolling(z_lookback).std()
    df['z'] = (df['spread'] - df['spread_mean']) / df['spread_std']

    r_y = df['Close_Y'].pct_change()
    r_x = df['Close_X'].pct_change()
    df['spread_return'] = r_y - beta.shift(1) * r_x

    position = np.zeros(len(df), dtype=int)
    holding_count = 0

    for i in range(1, len(df)):
        z_prev = df['z'].iloc[i - 1]
        prev = position[i - 1]

        if np.isnan(z_prev):
            position[i] = 0
            continue

        if prev != 0:
            holding_count += 1
            if abs(z_prev) > stop_z:
                position[i] = 0
                holding_count = 0
                continue
            if holding_count >= max_holding:
                position[i] = 0
                holding_count = 0
                continue
            if abs(z_prev) < exit_z:
                position[i] = 0
                holding_count = 0
                continue
            position[i] = prev
            continue

        if z_prev > entry:
            position[i] = -1
            holding_count = 0
        elif z_prev < -entry:
            position[i] = 1
            holding_count = 0
        else:
            position[i] = 0

    df['position'] = position
    df['position_size'] = np.where(position != 0, 1.0, 0.0)
    df['stop_loss'] = 0.0
    df['strategy_returns'] = df['spread_return']

    indicator_cols = ['beta', 'alpha', 'spread', 'z']

    df['position'] = df['position'].fillna(0).astype(int)
    df['position_size'] = df['position_size'].fillna(0.0)
    df['stop_loss'] = df['stop_loss'].fillna(0.0)
    df['strategy_returns'] = df['strategy_returns'].fillna(0.0)

    return df, indicator_cols

---
## Run Walk-Forward
Simulates how the strategy would have performed if re-optimised periodically in live trading, and exposes whether good IS performance survives unseen data.

**Folds Setup**
| Parameter | Description | Guidance |
|---|---|---|
| `TRAIN_BARS` | Bars per training window | 300–500 for daily pairs; needs enough bars for OLS + z-score warmup + meaningful trades |
| `TEST_BARS` | Bars per test window | 60–120 for daily; shorter than single-asset because spread regimes shift faster |
| `BURNIN_BARS` | Bars prepended to test for indicator warmup | ≥ `lookback` upper bound (e.g. 200) so OLS + z-score are initialised before first OOS bar |
| `N_TRIALS` | Optuna trials per fold | 500–800; pairs have more parameters than typical single-asset strategies |
| `COST` | Round-trip cost per trade | `0.001` = 0.1%; note this is applied to the *spread trade* (both legs combined) |

**Score and Rejection** — default `score_fn` uses weighted Sharpe/Calmar/Return. For pairs, the `reject_fn` should filter aggressively on trade count (spreads trade less frequently than directional strategies) and profit factor.

> Pay attention to the **degradation ratio** — OOS/IS Sharpe reveals overfitting. For pairs strategies, WFE < 0.3 often means the cointegration relationship is non-stationary across the test period.


In [52]:
# ── walk-forward windows ──────────────────────────────────────────────────────
TRAIN_BARS  = 500
TEST_BARS   = 126
BURNIN_BARS = 200    # >= max(lookback range) so OLS is warm before first OOS bar
N_TRIALS    = 800
COST        = 0.001

# ── SCORING FUNCTION ──────────────────────────────────────────────────────────
def score_fn(m):
    SHARPE_MAX = 2.5
    CALMAR_MAX = 70.0
    RETURN_MAX = 15.0

    calmar = m['total_return'] / abs(m['max_drawdown']) if m['max_drawdown'] != 0 else 0

    s = np.clip(m['sharpe_ratio']  / SHARPE_MAX, 0, 1)
    c = np.clip(calmar             / CALMAR_MAX, 0, 1)
    r = np.clip(m['total_return']  / RETURN_MAX, 0, 1)

    return 0.50 * s + 0.30 * c + 0.20 * r

# ── REJECTION CRITERIA ────────────────────────────────────────────────────────
def reject_fn(m):
    if m is None:                      return True
    if m['num_trades']    < 5:         return True
    if m['win_rate']      < 0.3:       return True
    if m['max_drawdown']  < -1:        return True
    if m['profit_factor'] < 0.7:       return True
    return False


results = walk_forward(
    df           = df,
    strategy_fn  = my_strategy,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    train_bars   = TRAIN_BARS,
    test_bars    = TEST_BARS,
    burnin_bars  = BURNIN_BARS,
    n_trials     = N_TRIALS,
    cost         = COST,
    score_fn     = score_fn,
    reject_fn    = reject_fn,
    save_csv     = None,
)


UPDATED WALK_FORWARD FILE IS RUNNING
Walk-forward: 7 fold(s)  train=500  test=126  burnin=200  trials=800
  Fold 1: train 2022-03-07 → 2023-07-19  | test 2023-07-20 → 2023-11-22
  Fold 2: train 2022-07-11 → 2023-11-22  | test 2023-11-23 → 2024-03-27
  Fold 3: train 2022-11-14 → 2024-03-27  | test 2024-03-28 → 2024-07-31
  Fold 4: train 2023-03-20 → 2024-07-31  | test 2024-08-01 → 2024-12-04
  Fold 5: train 2023-07-24 → 2024-12-04  | test 2024-12-05 → 2025-04-09
  Fold 6: train 2023-11-27 → 2025-04-09  | test 2025-04-10 → 2025-08-13
  Fold 7: train 2024-04-01 → 2025-08-13  | test 2025-08-14 → 2025-12-17

────────────────────────────────────────────────────────────
Fold 1/7  train: 2022-03-07 → 2023-07-19  test: 2023-07-20 → 2023-11-22


  0%|          | 0/800 [00:00<?, ?it/s]


  IS  → Sharpe: 3.64  Return: 238.60%  DD: -6.58%  Calmar: 21.81  Trades: 51
  OOS → Sharpe: 1.80  Return: 51.18%  DD: -19.10%  Calmar: 12.10  Trades: 13

  Best params: {'lookback': 150, 'z_lookback': 25, 'entry': 1.1014758190779643, 'exit_z': 1.2464078541363588, 'stop_z': 4.697498000254637, 'max_holding': 23}

────────────────────────────────────────────────────────────
Fold 2/7  train: 2022-07-11 → 2023-11-22  test: 2023-11-23 → 2024-03-27


  0%|          | 0/800 [00:00<?, ?it/s]


  IS  → Sharpe: 2.11  Return: 63.11%  DD: -7.01%  Calmar: 6.12  Trades: 14
  OOS → Sharpe: -1.61  Return: -5.30%  DD: -6.60%  Calmar: -2.21  Trades: 1

  Best params: {'lookback': 131, 'z_lookback': 55, 'entry': 2.026494344906071, 'exit_z': 1.292844715269772, 'stop_z': 4.590826930771029, 'max_holding': 6}

────────────────────────────────────────────────────────────
Fold 3/7  train: 2022-11-14 → 2024-03-27  test: 2024-03-28 → 2024-07-31


...


  IS  → Sharpe: 1.80  Return: 74.32%  DD: -9.60%  Calmar: 5.21  Trades: 7
  OOS → Sharpe: 0.89  Return: 5.71%  DD: -5.19%  Calmar: 3.36  Trades: 3

  Best params: {'lookback': 96, 'z_lookback': 78, 'entry': 2.1932608857532045, 'exit_z': 0.7506050574909098, 'stop_z': 4.262853751042498, 'max_holding': 24}

────────────────────────────────────────────────────────────
Fold 6/7  train: 2023-11-27 → 2025-04-09  test: 2025-04-10 → 2025-08-13


  0%|          | 0/800 [00:00<?, ?it/s]


  IS  → Sharpe: 2.82  Return: 154.79%  DD: -9.56%  Calmar: 10.25  Trades: 16
  OOS → Sharpe: -2.16  Return: -32.26%  DD: -37.76%  Calmar: -1.79  Trades: 3

  Best params: {'lookback': 85, 'z_lookback': 34, 'entry': 1.944928537706289, 'exit_z': 1.1073796424788012, 'stop_z': 3.732638272110156, 'max_holding': 22}

────────────────────────────────────────────────────────────
Fold 7/7  train: 2024-04-01 → 2025-08-13  test: 2025-08-14 → 2025-12-17


  0%|          | 0/800 [00:00<?, ?it/s]


  IS  → Sharpe: 2.37  Return: 159.02%  DD: -14.56%  Calmar: 6.89  Trades: 66
  OOS → Sharpe: 0.44  Return: 3.18%  DD: -10.48%  Calmar: 0.91  Trades: 30

  Best params: {'lookback': 80, 'z_lookback': 25, 'entry': 1.0322975310417082, 'exit_z': 1.4555620644582232, 'stop_z': 4.447312644350569, 'max_holding': 30}

════════════════════════════════════════════════════════════
WALK-FORWARD SUMMARY
════════════════════════════════════════════════════════════

Out-of-sample across 7 fold(s):
  Avg Sharpe:       0.32
  Avg Return:       8.6%
  Avg Max Drawdown: -12.4%
  Avg Calmar:       4.51
  Avg Trades/fold:  8
  Folds profitable: 4/7
  Sharpe OOS/IS:    0.14  (poor)

────────────────────────────────────────────────────────────
COMBINED OOS: 2023-07-20 → 2025-12-17  (868 bars)
  Return:        45.39%
  Sharpe:        0.57
  Max Drawdown:  -41.50%
  Calmar:        0.41
  Profit Factor: 0.63
  Win Rate:      49.06%
  Num Trades:    53


---
## Granular Results and Parameter Stability

Per-fold IS vs OOS performance. Each row is one fold — compare `train_*` vs `test_*` columns to assess overfitting.

| Column | Description |
|---|---|
| `*_sharpe` `*_return` `*_drawdown` `*_calmar` | Core performance metrics |
| `*_trades` `*_winrate` `*_profit_factor` | Trade statistics |
| `optuna_score` | Best score achieved on training window |
| `param_*` | Best parameter values per fold e.g. `param_lookback`, `param_entry` |

**Consensus Parameters** — the engine determines stability using the coefficient of variation (CV): the standard deviation of a parameter's best values across all folds divided by their median.

> CV < 0.15: the optimizer consistently picks a similar value — safe to anchor. High CV means the parameter is period-sensitive and should stay free (or the relationship is non-stationary).

**Pairs-specific stability notes:**
- If `lookback` has high CV but `entry`/`exit_z` are stable, the hedge ratio is adapting to structural shifts while the trading logic is consistent — this is often healthy.
- If `entry` has high CV, the strategy is chasing different spread volatility regimes each fold — consider normalising the entry threshold differently.


In [53]:
# ── fold summary table ────────────────────────────────────────────────────────
display_cols = [
    'train_start', 'train_end',
    'test_start', 'test_end',
    'train_sharpe', 'test_sharpe',
    'train_return', 'test_return',
    'train_drawdown', 'test_drawdown',
    'test_trades',  'test_winrate', 'optuna_score',
]
display(results['results_df'][display_cols].round(2))

# ── parameter stability ───────────────────────────────────────────────────────
stability = results['stability_df'].copy()
stability['stable'] = stability['stable'].map({True: '✓', False: ''})
stability['fixed']  = stability['fixed'].map({True: '✓', False: ''})
stability = stability[['param', 'median', 'std', 'cv', 'stable', 'fixed']].round(2)
display(stability.sort_values('cv'))

# ── consensus params ──────────────────────────────────────────────────────────
stable = results['stability_df'][results['stability_df']['cv'] < 0.15]

print('Stable parameters (CV < 0.15) — copy into FIXED_PARAMS:')
for _, row in stable.iterrows():
    v = results['consensus_params'][row['param']]
    v_fmt = int(round(v)) if isinstance(v, float) and v == int(v) else round(v, 4) if isinstance(v, float) else v
    print(f"    '{row['param']}': {v_fmt},")

print('\nConsensus parameters (median across folds):')
for k, v in results['consensus_params'].items():
    print(f'  {k:<30} = {round(v, 2) if isinstance(v, float) else v}')


,train_start,train_end,test_start,test_end,train_sharpe,test_sharpe,train_return,test_return,train_drawdown,test_drawdown,test_trades,test_winrate,optuna_score
0,2022-03-07,2023-07-19,2023-07-20,2023-11-22,3.64,1.80,2.39,0.51,-0.07,-0.19,13,0.46,0.69
1,2022-07-11,2023-11-22,2023-11-23,2024-03-27,2.11,-1.61,0.63,-0.05,-0.07,-0.07,1,1.00,0.47
2,2022-11-14,2024-03-27,2024-03-28,2024-07-31,1.20,0.00,0.23,0.00,-0.07,0.00,0,0.00,0.26
3,2023-03-20,2024-07-31,2024-08-01,2024-12-04,2.15,2.87,0.71,0.38,-0.08,-0.08,3,0.67,0.48
4,2023-07-24,2024-12-04,2024-12-05,2025-04-09,1.80,0.89,0.74,0.06,-0.10,-0.05,3,0.67,0.40
5,2023-11-27,2025-04-09,2025-04-10,2025-08-13,2.82,-2.16,1.55,-0.32,-0.10,-0.38,3,0.33,0.59
6,2024-04-01,2025-08-13,2025-08-14,2025-12-17,2.37,0.44,1.59,0.03,-0.15,-0.10,30,0.47,0.54


,param,median,std,cv,stable,fixed
4,stop_z,4.26,0.34,0.08,✓,
0,lookback,126.00,25.68,0.20,,
3,exit_z,1.25,0.32,0.26,,
2,entry,2.03,0.54,0.27,,
5,max_holding,22.00,8.53,0.39,,
1,z_lookback,51.00,21.04,0.41,,


Stable parameters (CV < 0.15) — copy into FIXED_PARAMS:
    'stop_z': 4.2629,

Consensus parameters (median across folds):
  lookback                       = 126
  z_lookback                     = 51
  entry                          = 2.03
  exit_z                         = 1.25
  stop_z                         = 4.26
  max_holding                    = 22


---
## Parameter Robustness Checks

### Plateau Analysis
Sweep each free parameter across its range while holding others at consensus (median) value, then evaluates the `score` at each point by backtesting over the entire lookback.

The stability table (CV across folds) tells you *"does the optimizer consistently pick the same value?"*  

Plateau analysis tells you *"if that value were slightly wrong, would performance collapse?"*  

**Plateau %** — what fraction of each parameter's range stays within 20% of peak score: >60% = `robust plateau`, 30–60% = `moderate`, <30% = `fragile` (consider anchoring). `N/A` means every sweep point failed rejection filters — the strategy is completely intolerant of movement on that dimension.

**Pairs-specific plateau notes:**
- `lookback` often shows a broad plateau (OLS is forgiving of window size) — if it's fragile, the cointegration itself may be weak.
- `entry` tends to have a narrower plateau than single-asset thresholds because the z-score distribution of a spread is more concentrated.
- A `stop_z` plateau that extends to the upper bound means the stop rarely triggers — consider removing it as a free parameter.

> Run time: `n_free_params` × `n_steps`

### Perturbation Test
Jitters all free parameters by ±5/10/20% of their range (50 random samples per offset range). Measures how much the score degrades vs the base.

Tests whether the optimum is a broad hill in `#free params`-D space or a narrow spike.

**>15% degradation at ±10%:** fragile optimum — consider reducing free parameters.


In [54]:
# ── 1-D sensitivity sweeps around consensus params ─────────────────────────
sweep_results = plateau_analysis(
    df           = df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = COST,
    n_steps      = 20,
)

# ── text verdicts ──────────────────────────────────────────────────────────
verdict_df = plateau_summary(
    sweep_results,
    base_params  = results['consensus_params'],
    stability_df = results['stability_df'],
    threshold    = 0.20,
)

# ── neighbourhood perturbation ────────────────────────────────────────────
perturb_df = perturbation_test(
    df           = df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = COST,
    pct_offsets  = (0.05, 0.10, 0.20),
    n_samples    = 50,
)



═══════════════════════════════════════════════════════════════════════════════════════════
PLATEAU ANALYSIS — PARAMETER ROBUSTNESS
═══════════════════════════════════════════════════════════════════════════════════════════
Parameter                 Consensus Peak Score  Plateau %  Fold CV Verdict                 
──────────────────────── ────────── ────────── ────────── ──────── ────────────────────────
entry                        2.0265      0.000      100.0%    0.265 Robust                  
stop_z                       4.2629      0.000      100.0%    0.080 Robust                  
lookback                        126      0.052       10.0%    0.204 FRAGILE                 
max_holding                      22      0.033       10.0%    0.388 FRAGILE                 
z_lookback                       51      0.004        8.3%    0.412 FRAGILE                 
exit_z                       1.2464      0.062        5.0%    0.255 FRAGILE                 

════════════════════════════════

### 1-D Sweep Charts
| Element | Meaning | Good | Bad |
|---------|---------|------|-----|
| **Blue curve** | Composite score at each value of the parameter, with all others held at consensus | Flat-topped curve — performance is insensitive to the exact value | Narrow spike — optimizer latched onto one specific value, everything nearby is worse |
| **Red dashed line** | Where the consensus value sits | On the flat top of the curve | On a steep slope or at the edge of a cliff |
| **Green dashed line** | Cutoff at 80% of peak score — the boundary between plateau and non-plateau | Blue curve stays above this line across most of the range | Blue curve dips below it quickly either side of the peak |
| **Green shading** | Plateau region — all values where the score stays within 20% of the peak | Wide green band spanning most of the range (robust) | Thin sliver or no shading at all (fragile/overfit) |

**Pairs-specific interpretation:**
- `lookback` sweep with a valley in the middle often means short windows capture noise, long windows miss regime changes — look for a stable plateau in between.
- `entry` sweep that peaks sharply at one value and collapses at ±0.2 means the spread distribution is very specific — the strategy is fragile to z-score calibration.



In [55]:
os.makedirs('outputs', exist_ok=True)

# ── visual sweep curves ───────────────────────────────────────────────────
plot_plateau_analysis(
    sweep_results    = sweep_results,
    consensus_params = results['consensus_params'],
    param_defs       = PARAM_DEFS,
    fixed_params     = FIXED_PARAMS,
    threshold        = 0.20,
    show             = False,
    save_html        = 'outputs/wf_plateau_analysis.html',
)


✓ Saved plateau analysis chart → outputs/wf_plateau_analysis.html


---
## Results Charts and Cost Stress Test

| Parameter | Description | Default |
|---|---|---|
| `show_fold_perf` | IS vs OOS bars for return, Sharpe, drawdown per fold | `False` |
| `show_param_evol` | Parameter evolution across folds with ±1 std bands | `False` |
| `show_oos_equity` | Combined OOS equity curve + drawdown with fold boundaries | `True` |
| `show_trades` | Overlay entry/exit markers on OOS equity chart | `False` |
| `benchmark_data` | DataFrame with `Close` column for comparison — for pairs, this is the raw spread return (untraded) | `None` |
| `save_html_dir` | Directory path to save charts as HTML files, or `None` | `None` |

**Cost Stress Test:** Re-run the combined OOS backtest at 1×, 1.5×, 2×, 3× the base cost. Pairs strategies trade both legs simultaneously so effective costs are higher — a strategy that collapses at 1.5× is not viable for live execution.

**Pairs-specific note on benchmark:** Unlike single-asset strategies where buy-and-hold is the natural benchmark, pairs strategies should be compared against the *untraded spread* (i.e., what happens if you just held the spread permanently). If the strategy can't beat the untraded spread after costs, the entry/exit logic adds no value.


In [56]:
plot_walk_forward_results(
    results          = results,
    param_defs       = PARAM_DEFS,
    fixed_params     = FIXED_PARAMS,
    benchmark_data   = None,        # no single-asset benchmark for pairs
    show             = True,
    save_html_dir    = None,
    show_fold_perf   = False,
    show_param_evol  = False,
    show_oos_equity  = True,
    show_trades      = False,
)

# ── transaction cost stress test ──────────────────────────────────────────
if results['oos_combined_df'] is not None:
    cost_df = cost_stress_test(
        oos_combined_df  = results['oos_combined_df'],
        cost_multipliers = (1.0, 1.5, 2.0, 3.0),
        base_cost        = COST,
    )
else:
    print('No combined OOS dataframe — skip cost stress test')



═══════════════════════════════════════════════════════════════════════════
TRANSACTION COST STRESS TEST
═══════════════════════════════════════════════════════════════════════════
    Cost   Mult   Sharpe     Return      MaxDD   Calmar       PF
──────── ────── ──────── ────────── ────────── ──────── ────────
  0.0010   1.0x     0.57     45.39%    -41.50%     0.41     0.63
  0.0015   1.5x     0.52     37.79%    -42.55%     0.34     0.63
  0.0020   2.0x     0.46     30.59%    -43.62%     0.27     0.62
  0.0030   3.0x     0.35     17.28%    -46.06%     0.15     0.61
